---
title: "Polars, the New Gold Standard for Scikit-Learn Workflows"
subtitle: Ditch the NumPy bottleneck and build end-to-end workflows.
abstract: |
  The transition from data manipulation to machine learning has historically been a point of friction, often requiring the conversion of structured DataFrames into "anonymous" NumPy arrays.While this transition is the foundation of the scikit-learn ecosystem, it frequently results in the loss of critical metadata, specifically feature names and data types. Traditionally, Pandas has been the primary tool for managing this workflow, but as datasets grow in complexity and scale, its single-threaded nature and memory overhead have become significant bottlenecks.
  <br>
  This article explores the modern alternative: Polars. We demonstrate how the native integration between Polars and scikit-learn (introduced in version 1.4) allows data scientists to maintain a "DataFrame-in, DataFrame-out" workflow. By leveraging the transform_output="polars" configuration, practitioners can perform high-speed, multi-threaded feature engineering in Rust-backed Polars while preserving column identities through every stage of a pipeline.   
  <br>
  We provide a comparative analysis of the benefits, such as automatic horizontal concatenation (hstack) and Apache Arrow memory efficiency, alongside the limitations, including the current lack of native support for Polars LazyFrames within fit/predict cycles. Readers will learn how to build a robust, name-aware machine learning pipeline that is significantly faster and more memory-efficient than traditional Pandas-based approaches.   
author: Jesus LM
date: 2026-03-16
date-format: MMM, YYYY
format:
  html:
    toc: true
    embed-resources: true
    theme: custom.scss
    execute: true
jupyter: python3
---

# Machine Learning Workflow

## Load dataset

In [1]:
# The magic line that keeps your column names
#from sklearn import set_config
#set_config(transform_output="polars")

In [83]:
import pandas as pd
import polars as pl

df = pl.read_csv('http://bit.ly/MLtrain')
df = df.drop_nulls(subset='Embarked')
df.head()

Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
i64,i64,str,str,f64,i64,i64,str,f64,str,str
0,3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S"""
1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C"""
1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S"""
1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S"""
0,3,"""Allen, Mr. William Henry""","""male""",35.0,0,0,"""373450""",8.05,null,"""S"""


In [84]:
X = df.select('Parch', 'Fare')

In [85]:
y = df.get_column('Survived')

In [86]:
X.shape

(889, 2)

In [87]:
y.shape

(889,)

## Model selection

In [88]:
from sklearn.linear_model import LogisticRegression
logreg = LogisticRegression(solver='liblinear', random_state=1)

In [89]:
from sklearn.model_selection import cross_val_score
cross_val_score(logreg, X, y, cv=3, scoring='accuracy').mean()

np.float64(0.6648239148239148)

In [90]:
logreg.fit(X, y)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multic

In [91]:
df_new = pl.read_csv('http://bit.ly/MLnewdata', n_rows=10)
df_new

Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
i64,str,str,f64,i64,i64,str,f64,str,str
3,"""Kelly, Mr. James""","""male""",34.5,0,0,"""330911""",7.8292,null,"""Q"""
3,"""Wilkes, Mrs. James (Ellen Need…","""female""",47.0,1,0,"""363272""",7.0,null,"""S"""
2,"""Myles, Mr. Thomas Francis""","""male""",62.0,0,0,"""240276""",9.6875,null,"""Q"""
3,"""Wirz, Mr. Albert""","""male""",27.0,0,0,"""315154""",8.6625,null,"""S"""
3,"""Hirvonen, Mrs. Alexander (Helg…","""female""",22.0,1,1,"""3101298""",12.2875,null,"""S"""
3,"""Svensson, Mr. Johan Cervin""","""male""",14.0,0,0,"""7538""",9.225,null,"""S"""
3,"""Connolly, Miss. Kate""","""female""",30.0,0,0,"""330972""",7.6292,null,"""Q"""
2,"""Caldwell, Mr. Albert Francis""","""male""",26.0,1,1,"""248738""",29.0,null,"""S"""
3,"""Abrahim, Mrs. Joseph (Sophie H…","""female""",18.0,0,0,"""2657""",7.2292,null,"""C"""


In [92]:
X_new = df_new.select(['Parch', 'Fare'])
X_new

Parch,Fare
i64,f64
0,7.8292
0,7.0
0,9.6875
0,8.6625
1,12.2875
0,9.225
0,7.6292
1,29.0
0,7.2292


In [93]:
logreg.predict(X_new)

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [94]:
logreg.predict_proba(X_new)

array([[0.69548825, 0.30451175],
       [0.69804976, 0.30195024],
       [0.68970366, 0.31029634],
       [0.69290181, 0.30709819],
       [0.67268411, 0.32731589],
       [0.691149  , 0.308851  ],
       [0.69610719, 0.30389281],
       [0.61680219, 0.38319781],
       [0.69734295, 0.30265705],
       [0.64274043, 0.35725957]])

In [95]:
logreg.predict_proba(X_new)[:, 1]

array([0.30451175, 0.30195024, 0.31029634, 0.30709819, 0.32731589,
       0.308851  , 0.30389281, 0.38319781, 0.30265705, 0.35725957])

In [96]:
logreg.get_params()

{'C': 1.0,
 'class_weight': None,
 'dual': False,
 'fit_intercept': True,
 'intercept_scaling': 1,
 'l1_ratio': 0.0,
 'max_iter': 100,
 'n_jobs': None,
 'penalty': 'deprecated',
 'random_state': 1,
 'solver': 'liblinear',
 'tol': 0.0001,
 'verbose': 0,
 'warm_start': False}

## Encoding Categorical Features

In [97]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output=False)

In [98]:
ohe.fit_transform(df.select('Embarked'))

array([[0., 0., 1.],
       [1., 0., 0.],
       [0., 0., 1.],
       ...,
       [0., 0., 1.],
       [1., 0., 0.],
       [0., 1., 0.]], shape=(889, 3))

In [99]:
ohe.categories_

[array(['C', 'Q', 'S'], dtype=object)]

In [100]:
ohe.fit_transform(df.select(['Embarked', 'Sex']))

array([[0., 0., 1., 0., 1.],
       [1., 0., 0., 1., 0.],
       [0., 0., 1., 1., 0.],
       ...,
       [0., 0., 1., 1., 0.],
       [1., 0., 0., 0., 1.],
       [0., 1., 0., 0., 1.]], shape=(889, 5))

In [101]:
ohe.categories_

[array(['C', 'Q', 'S'], dtype=object), array(['female', 'male'], dtype=object)]

## ColumnTransformer and Pipeline

In [102]:
cols = ['Parch', 'Fare', 'Embarked', 'Sex']
X = df.select(cols)

In [103]:
X

Parch,Fare,Embarked,Sex
i64,f64,str,str
0,7.25,"""S""","""male"""
0,71.2833,"""C""","""female"""
0,7.925,"""S""","""female"""
0,53.1,"""S""","""female"""
0,8.05,"""S""","""male"""
…,…,…,…
0,13.0,"""S""","""male"""
0,30.0,"""S""","""female"""
2,23.45,"""S""","""female"""


In [104]:
ohe = OneHotEncoder()

In [105]:
from sklearn.compose import make_column_transformer

ct = make_column_transformer(
    (ohe, ['Embarked', 'Sex']),
    remainder='drop')

In [106]:
ct.fit_transform(X)

array([[0., 0., 1., 0., 1.],
       [1., 0., 0., 1., 0.],
       [0., 0., 1., 1., 0.],
       ...,
       [0., 0., 1., 1., 0.],
       [1., 0., 0., 0., 1.],
       [0., 1., 0., 0., 1.]], shape=(889, 5))

In [107]:
ct = make_column_transformer(
    (ohe, ['Embarked', 'Sex']),
    remainder='passthrough')

In [108]:
ct.fit_transform(X)

array([[ 0.    ,  0.    ,  1.    , ...,  1.    ,  0.    ,  7.25  ],
       [ 1.    ,  0.    ,  0.    , ...,  0.    ,  0.    , 71.2833],
       [ 0.    ,  0.    ,  1.    , ...,  0.    ,  0.    ,  7.925 ],
       ...,
       [ 0.    ,  0.    ,  1.    , ...,  0.    ,  2.    , 23.45  ],
       [ 1.    ,  0.    ,  0.    , ...,  1.    ,  0.    , 30.    ],
       [ 0.    ,  1.    ,  0.    , ...,  1.    ,  0.    ,  7.75  ]],
      shape=(889, 7))

In [109]:
ct.get_feature_names_out()

array(['onehotencoder__Embarked_C', 'onehotencoder__Embarked_Q',
       'onehotencoder__Embarked_S', 'onehotencoder__Sex_female',
       'onehotencoder__Sex_male', 'remainder__Parch', 'remainder__Fare'],
      dtype=object)

In [110]:
ct = make_column_transformer(
    (ohe, ['Embarked', 'Sex']),
    ('passthrough', ['Parch', 'Fare']))

In [111]:
ct.fit_transform(X)

array([[ 0.    ,  0.    ,  1.    , ...,  1.    ,  0.    ,  7.25  ],
       [ 1.    ,  0.    ,  0.    , ...,  0.    ,  0.    , 71.2833],
       [ 0.    ,  0.    ,  1.    , ...,  0.    ,  0.    ,  7.925 ],
       ...,
       [ 0.    ,  0.    ,  1.    , ...,  0.    ,  2.    , 23.45  ],
       [ 1.    ,  0.    ,  0.    , ...,  1.    ,  0.    , 30.    ],
       [ 0.    ,  1.    ,  0.    , ...,  1.    ,  0.    ,  7.75  ]],
      shape=(889, 7))

## Pipeline chaining

In [112]:
from sklearn.pipeline import make_pipeline
pipe = make_pipeline(ct, logreg)

In [113]:
pipe.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('columntransformer', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('onehotencoder', ...), ('passthrough', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the out

In [114]:
X_t = ct.fit_transform(X)
logreg.fit(X_t, y)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multic

In [115]:
print(X.shape)
print(X_t.shape)

(889, 4)
(889, 7)


In [118]:
data_new = {
    "Parch": [0, 0, 0, 0, 1, 0, 0, 1, 0, 0],
    "Fare": [7.8292, 7.0000, 9.6875, 8.6625, 12.2875, 9.2250, 7.6292, 29.0000, 7.2292, 24.1500],
    "Embarked": ["Q", "S", "Q", "S", "S", "S", "Q", "S", "C", "S"],
    "Sex": ["male", "female", "male", "male", "female", "male", "female", "male", "female", "male"]
}

X_new = pl.DataFrame(data_new)

In [119]:
pipe.predict(X_new)

array([0, 1, 0, 0, 1, 0, 1, 0, 1, 0])

In [120]:
X_new_t = ct.transform(X_new)
logreg.predict(X_new_t)

array([0, 1, 0, 0, 1, 0, 1, 0, 1, 0])

In [121]:
print(X_new.shape)
print(X_new_t.shape)

(10, 4)
(10, 7)


In [122]:
ct = make_column_transformer(
    (ohe, ['Embarked', 'Sex']),
    ('drop', ['Fare']),
    remainder='passthrough')
ct.fit_transform(X)

array([[0., 0., 1., 0., 1., 0.],
       [1., 0., 0., 1., 0., 0.],
       [0., 0., 1., 1., 0., 0.],
       ...,
       [0., 0., 1., 1., 0., 2.],
       [1., 0., 0., 0., 1., 0.],
       [0., 1., 0., 0., 1., 0.]], shape=(889, 6))

In [123]:
ct = make_column_transformer(
    (ohe, ['Embarked', 'Sex']),
    ('passthrough', ['Parch']),
    remainder='drop')
ct.fit_transform(X)

array([[0., 0., 1., 0., 1., 0.],
       [1., 0., 0., 1., 0., 0.],
       [0., 0., 1., 1., 0., 0.],
       ...,
       [0., 0., 1., 1., 0., 2.],
       [1., 0., 0., 0., 1., 0.],
       [0., 1., 0., 0., 1., 0.]], shape=(889, 6))

# Contact